# Table 3 of Part 1: descriptor/embedding MLPs, GNNs and GNN–descriptor hybrids (full scale, 5 seeds)

Every model in Table 3 is trained and evaluated here on one label set: the common configuration of
10 water mass fractions × 3 temperatures per BIMOG molecule, with the SHA-256 hash split (56 held-out molecules)
and a fixed 20 % interpolation split. The labels are identical to those of Model A_emb in
`02_representation_emb_reruns.ipynb`.

| Model | Description |
|---|---|
| A_emb | MLP with the learned subgroup embedding (R_emb) |
| A_desc | MLP with hand-crafted descriptors (R_desc) |
| B | GNN (3-layer GINE, mean + sum pooling, 64-dim embedding) trained on BIMOG only |
| C | Same GNN pretrained on a synthetic AIOMFAC-labelled pool, then fine-tuned on BIMOG with a new head |
| D | GNN embedding ⊕ R_desc, trained end to end |
| E | As D, with the GNN initialized from the pretrained encoder of C |
| F | Frozen pretrained encoder ⊕ R_desc behind a dropout-regularized head |
| G | As F with a frozen, randomly initialized encoder (control) |

**Synthetic pretraining pool.** The pool is generated from chain lengths 1–10 × {linear, methyl-branched, ring}
× 13 monofunctional templates, plus 6 difunctional templates on linear chains, and filtered to
AIOMFAC-decomposable molecules. The paper's original pool used the same counts (10 × 3 × 13 + 6 templates,
346 molecules), but its exact template list was not archived. Ten templates are those of the v1 notebook; the
remaining three monofunctional and six difunctional templates are defined here.

The GNN is implemented in plain PyTorch, with the same architecture as the v1 notebook (which used
`torch_geometric`).

In [1]:
import json, time, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
from rdkit import Chem, RDLogger
from aiomfac_py import ActivityModel, Component
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py.params import load_subgroup_params
import aiomfac_py.s2as._mapping as _m
_m.print = lambda *a, **k: None; RDLogger.DisableLog("rdApp.*")
sg = load_subgroup_params(); WATER = Component(1, "Water", ((16, 1),))
hash100 = lambda s: int(hashlib.sha256(s.encode()).hexdigest(), 16) % 100
fmt = lambda s: f"{np.mean(s):.3f} ± {np.std(s, ddof=1):.3f}" if len(s) > 1 else f"{np.mean(s):.3f}"
T0 = time.time()

def featurize(name, smi, cat):
    r = smiles_to_components([smi], water_as_component1=True)
    if r.removed: return None
    try:
        if not (ActivityModel(r.components).evaluate([0.9, 0.1], 298.15, basis="mass").gamma_neutral > 0).all(): return None
    except Exception: return None
    mol = Chem.MolFromSmiles(smi); mh = Chem.AddHs(mol); sym = [a.GetSymbol() for a in mh.GetAtoms()]; nc = max(sym.count("C"), 1)
    return {"name": name, "smiles": smi, "canon": Chem.MolToSmiles(mol), "cat": cat, "subgroups": r.components[1].subgroups,
            "M": sum(a.GetMass() for a in mh.GetAtoms()), "oc": sym.count("O") / nc, "nc": sym.count("N") / nc,
            "hc": sym.count("H") / nc, "unm": int(r.n_unmatched_atoms[0])}

def bimog_cat(d):
    n, h = d["#N"] != 0, d["#Hal"] != 0
    return "n_and_hal" if n and h else "n_only" if n else "hal_only" if h else "cho"

bim_raw = json.load(open("bimog_unified.json"))
bimog = [o for d in bim_raw if (o := featurize(d["name"], d["smiles"], bimog_cat(d)))]
bcanon = {o["canon"] for o in bimog}

synth = []
add = lambda s, n: synth.append((n, s))
for n in range(1, 11): add("C" * n + "N", f"amine1-C{n}")
for n in range(1, 7): add("C" * n + "N" + "C" * n, f"amine2s-C{n}")
for n in range(1, 6):
    for m in range(n + 1, 7): add("C" * n + "N" + "C" * m, f"amine2-C{n}C{m}")
for n in range(1, 9): add("C" * n + "[N+](=O)[O-]", f"nitro-C{n}")
add("c1ccccc1[N+](=O)[O-]", "nitrobenzene")
for n in range(1, 5): add("C" * n + "c1ccccc1[N+](=O)[O-]", f"nitroarom-C{n}")
for n in range(1, 11): add("C" * n + "O[N+](=O)[O-]", f"nitrate-C{n}")
for n in range(2, 8):
    add("C" * (n - 1) + "C(O)O[N+](=O)[O-]", f"hn-a-C{n}"); add("OC" + "C" * (n - 2) + "CO[N+](=O)[O-]", f"hn-b-C{n}")
for n in range(2, 8): add("NC" + "C" * (n - 2) + "CO", f"aminoalc-C{n}")
for s, n in [("CC(C)N", "isopropylamine"), ("CC(C)(C)N", "tBuNH2"), ("CC(C)CN", "iBuNH2"), ("CC(N)CC", "sBuNH2"),
             ("C1CCC(CC1)N", "cyhexNH2"), ("C1CCC(C1)N", "cypentNH2"), ("CC(C)(C)CN", "neopentNH2"), ("COCCN", "MeOEtNH2"),
             ("CCOCCCN", "EtOPrNH2"), ("CC(=O)CCN", "aminobutanone"), ("CCOC(=O)CCN", "Et-aminopropanoate"),
             ("NCCOCCN", "bisaminoethylether"), ("NCCCCN", "putrescine"), ("NCCCCCN", "cadaverine"), ("NCCCCCCN", "HMDA"),
             ("NC(C)CN", "diaminopropane"), ("CC(C)[N+](=O)[O-]", "2-nitropropane"), ("C1CCC(CC1)[N+](=O)[O-]", "nitrocyclohexane"),
             ("CC(C)(C)[N+](=O)[O-]", "tBuNO2"), ("CC(=O)CC[N+](=O)[O-]", "nitrobutanone"), ("COCC[N+](=O)[O-]", "MeO-nitroethane"),
             ("CC(=O)CO[N+](=O)[O-]", "nitrooxypropanone"), ("COCCO[N+](=O)[O-]", "MeOEt-nitrate"), ("CC(C)O[N+](=O)[O-]", "iPr-nitrate"),
             ("C1CCC(CC1)O[N+](=O)[O-]", "cyhex-nitrate"), ("CC(C)NC(C)C", "diisopropylamine"), ("C1CCNCC1", "piperidine"),
             ("C1CCNC1", "pyrrolidine")]:
    add(s, n)
synth = [o for n, s in synth if (o := featurize(n, s, "synth_n"))]

mraw = pd.read_csv("mcm_v331_species.tsv", sep="\t", comment="*"); mcm = []
for _, r in mraw.iterrows():
    mol = Chem.MolFromSmiles(str(r.Smiles))
    if (mol is None or str(r.Excited).lower() == "true" or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms())
            or not {a.GetSymbol() for a in mol.GetAtoms()} <= {"C", "H", "O", "N"} or mol.GetNumHeavyAtoms() < 2
            or "C" not in {a.GetSymbol() for a in mol.GetAtoms()}): continue
    o = featurize(r.Name, str(r.Smiles), "mcm")
    if o and o["unm"] == 0 and o["canon"] not in bcanon: mcm.append(o)

POOL = bimog + synth + mcm
SRC = np.array(["bimog"] * len(bimog) + ["synth"] * len(synth) + ["mcm"] * len(mcm))
CAT = np.array([o["cat"] for o in POOL]); H = np.array([hash100(o["smiles"]) for o in POOL]); TEST = H < 15
VOCAB = sorted({int(s) for o in POOL for s, q in o["subgroups"]})
def counts(o):
    v = np.zeros(len(VOCAB), np.float32)
    for s, q in o["subgroups"]: v[VOCAB.index(int(s))] = q
    return v
CNT = np.array([counts(o) for o in POOL])
print(f"BIMOG {len(bimog)}, synthetic N {len(synth)}, MCM {len(mcm)}; vocabulary {len(VOCAB)} subgroups ({time.time()-T0:.0f}s)")

BIMOG 368, synthetic N 100, MCM 3337; vocabulary 55 subgroups (10s)


In [2]:
def label(ids, n_comp=20, n_temp=4, seed=0):
    '''AIOMFAC labels for the molecules in ids (Part I sampling). Returns molecule index, (w, T_norm), y.'''
    rng = np.random.RandomState(seed); rows = []
    for oi in ids:
        m = ActivityModel([WATER, Component(2, "o", tuple(POOL[oi]["subgroups"]))]); e = np.linspace(0.02, 0.98, n_comp + 1)
        for w in rng.uniform(e[:-1], e[1:]):
            for T in rng.uniform(273.15, 313.15, n_temp):
                try:
                    lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
                    if np.isfinite(lg[:2]).all(): rows.append((oi, w, (T - 293.15) / 20, lg[0], lg[1]))
                except Exception: pass
    a = np.array(rows)
    return a[:, 0].astype(int), a[:, 1:3].astype(np.float32), a[:, 3:5].astype(np.float32)

PM, EXTRA, Y = label(np.arange(len(POOL)))                     # 20 x 4 labels for every molecule
print(f"{len(Y)} labelled points ({time.time()-T0:.0f}s)")


class EmbedNet(nn.Module):
    def __init__(self, n_vocab, n_extra, n_out=2, emb=16):
        super().__init__()
        self.n_vocab = n_vocab; self.emb = nn.Linear(n_vocab, emb, bias=False)
        self.head = nn.Sequential(nn.Linear(emb + n_extra, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                                  nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, n_out))
    def forward(self, x): return self.head(torch.cat([self.emb(x[:, :self.n_vocab]), x[:, self.n_vocab:]], 1))


def train_C(Xc, Xe, Yt, train_pts, seed, interp_split=False, max_epochs=500, patience=30, batch=256):
    '''Xc: per-point counts (unscaled); Xe: per-point extra inputs (standardized here). Returns predict(Xc, Xe).'''
    tr = np.array(train_pts)
    if interp_split:                                     # Part I/II: fixed 20 % interpolation split, val = first 10 %
        p = np.random.RandomState(0).permutation(len(tr)); fit = tr[p[int(0.2 * len(tr)):]]
        nv = int(0.1 * len(fit)); val, fit = fit[:nv], fit[nv:]
    else:                                                # learning-curve style: seeded 10 % validation
        r0 = np.random.RandomState(seed); tr = tr.copy(); r0.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    em, es = Xe[fit].mean(0), Xe[fit].std(0); es = np.where(es < 1e-6, 1.0, es)
    ym, ys = Yt[fit].mean(0), Yt[fit].std(0) + 1e-6
    X = torch.tensor(np.concatenate([Xc, (Xe - em) / es], 1).astype(np.float32)); Yn = torch.tensor((Yt - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = EmbedNet(Xc.shape[1], Xe.shape[1], Yt.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    L = nn.MSELoss(); best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(net(X[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(X[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    def predict(xc, xe):
        with torch.no_grad():
            return net(torch.tensor(np.concatenate([xc, (xe - em) / es], 1).astype(np.float32))).numpy() * ys + ym
    return predict, ep + 1


def pts_of(mask_mol, pm=None):
    pm = PM if pm is None else pm
    return np.where(mask_mol[pm])[0]

304400 labelled points (34s)


In [3]:
SEEDS = range(5)
isB = SRC == "bimog"; bids = np.where(isB)[0]
PM2, EX2, Y2 = label(bids, n_comp=10, n_temp=3)                 # identical to notebook 02, section 2
gen_pts = np.where(TEST[PM2])[0]; tr_all = np.where(~TEST[PM2])[0]
p0 = np.random.RandomState(0).permutation(len(tr_all))
interp_pts = tr_all[p0[:int(0.2 * len(tr_all))]]; fit_all = tr_all[p0[int(0.2 * len(tr_all)):]]
nv = int(0.1 * len(fit_all)); val_pts, fit_pts = fit_all[:nv], fit_all[nv:]
MGS = sorted({sg.main_group(int(s)) for i in bids for s, q in POOL[i]["subgroups"]})
def desc(o):
    c, tq = np.zeros(len(MGS)), 0
    for s, q in o["subgroups"]:
        c[MGS.index(sg.main_group(int(s)))] += q; tq += q
    return [*(c / max(tq, 1)), o["oc"], o["nc"], np.log(o["M"]), np.log1p(tq)]
DESC = np.array([desc(o) if SRC[i] == "bimog" else np.zeros(len(MGS) + 4) for i, o in enumerate(POOL)], np.float32)
print(f"{len(Y2)} points: fit {len(fit_pts)}, val {len(val_pts)}, interp {len(interp_pts)}, gen {len(gen_pts)}; R_desc dim {DESC.shape[1]}")

11040 points: fit 6740, val 748, interp 1872, gen 1680; R_desc dim 22


## 1) Models A_emb and A_desc (MLPs)

In [4]:
class Head(nn.Module):
    def __init__(self, n_in, h=128, L=3, drop=0.0):
        super().__init__()
        layers, d = [], n_in
        for _ in range(L):
            layers += [nn.Linear(d, h), nn.ReLU()] + ([nn.Dropout(drop)] if drop > 0 else []); d = h
        self.f = nn.Sequential(*layers, nn.Linear(d, 2))
    def forward(self, x): return self.f(x)

def run_mlp(X, seed, unscaled_cols=0, max_epochs=300, patience=25, batch=256):
    xm, xs = X[fit_pts].mean(0), X[fit_pts].std(0); xs = np.where(xs < 1e-3, 1.0, xs)
    if unscaled_cols: xm[:unscaled_cols] = 0; xs[:unscaled_cols] = 1
    ym, ys = Y2[fit_pts].mean(0), Y2[fit_pts].std(0) + 1e-6
    Xn = torch.tensor(((X - xm) / xs).astype(np.float32)); Yn = torch.tensor((Y2 - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = Head(X.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit_pts), torch.tensor(val_pts)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit_pts)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(net(Xn[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(Xn[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    with torch.no_grad(): P = net(Xn).numpy() * ys + ym
    E = np.abs(P - Y2)
    return {"gen_w": E[gen_pts, 0].mean(), "gen_org": E[gen_pts, 1].mean(), "interp_org": E[interp_pts, 1].mean(), "epochs": ep + 1}

res = []
for seed in SEEDS:
    pred, ep = train_C(CNT[PM2], EX2, Y2, np.where(~TEST[PM2])[0], seed, interp_split=True, max_epochs=300, patience=25)
    E = np.abs(pred(CNT[PM2], EX2) - Y2)
    res.append({"model": "A_emb", "seed": seed, "gen_w": E[gen_pts, 0].mean(), "gen_org": E[gen_pts, 1].mean(), "interp_org": E[interp_pts, 1].mean(), "epochs": ep})
    res.append({"model": "A_desc", "seed": seed, **run_mlp(np.concatenate([DESC[PM2], EX2], 1), seed)})
print(pd.DataFrame(res).groupby("model")[["gen_w", "gen_org", "interp_org"]].agg(fmt), f"({time.time()-T0:.0f}s)")

                gen_w        gen_org     interp_org
model                                              
A_desc  0.032 ± 0.004  0.322 ± 0.027  0.088 ± 0.008
A_emb   0.021 ± 0.001  0.181 ± 0.009  0.085 ± 0.006 (69s)


## 2) Synthetic pretraining pool

In [5]:
MONO = {"alcohol": lambda c: c + "O", "ketone": lambda c: "CC(=O)" + c[1:] if len(c) > 1 else None,
        "ester_methyl": lambda c: c + "C(=O)OC", "ether_methyl": lambda c: c + "OC", "amine": lambda c: c + "N",
        "nitro": lambda c: c + "[N+](=O)[O-]", "acid": lambda c: c + "C(=O)O", "aldehyde": lambda c: c + "C=O",
        "chloro": lambda c: c + "Cl", "diol": lambda c: c + "O" if len(c) < 2 else "OC" + c[1:] + "O",
        # three templates added for the full-scale pool
        "nitrate": lambda c: c + "O[N+](=O)[O-]", "hydroperoxide": lambda c: c + "OO", "ether_ethyl": lambda c: c + "OCC"}
DI = {"hydroxy_acid": lambda c: "OC" + c[1:] + "C(=O)O", "keto_acid": lambda c: "CC(=O)" + c[1:] + "C(=O)O" if len(c) > 1 else None,
      "hydroxy_ketone": lambda c: "OC" + c[1:] + "C(C)=O", "diacid": lambda c: "OC(=O)" + c + "C(=O)O",
      "hydroxy_nitrate": lambda c: "OC" + c[1:] + "O[N+](=O)[O-]", "keto_aldehyde": lambda c: "CC(=O)" + c[1:] + "C=O" if len(c) > 1 else None}
def chain(n, br):
    if br == "linear": return "C" * n
    if br == "methyl_branch": return "C" * (n // 2) + "C(C)" + "C" * (n - n // 2 - 1) if n >= 3 else None
    if br == "ring": return "C1" + "C" * (n - 2) + "C1" if n >= 3 else None
seen, synth_cands = set(bcanon), []                         # exclude BIMOG molecules (avoid test leakage)
for n in range(1, 11):
    for br in ("linear", "methyl_branch", "ring"):
        c = chain(n, br)
        if c is None: continue
        temps = list(MONO.items()) + (list(DI.items()) if br == "linear" else [])
        for nm, fn in temps:
            try: smi = fn(c)
            except Exception: smi = None
            if not smi: continue
            mol = Chem.MolFromSmiles(smi)
            if mol is None: continue
            can = Chem.MolToSmiles(mol)
            if can in seen: continue
            seen.add(can); synth_cands.append((f"{nm}_C{n}_{br}", can))
SYN = [o for nm, s in synth_cands if (o := featurize(nm, s, "synth_pretrain"))]
print(f"synthetic pretraining pool: {len(SYN)} AIOMFAC-decomposable molecules of {len(synth_cands)} candidates "
      f"(BIMOG molecules excluded)")
rng = np.random.RandomState(42); srows = []
for k, o in enumerate(SYN):
    m = ActivityModel([WATER, Component(2, "o", tuple(o["subgroups"]))]); e = np.linspace(0.02, 0.98, 11)
    for w in rng.uniform(e[:-1], e[1:]):
        for T in rng.uniform(273.15, 313.15, 2):
            try:
                lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
                if np.isfinite(lg[:2]).all(): srows.append((k, w, (T - 293.15) / 20, lg[0], lg[1]))
            except Exception: pass
srows = np.array(srows); SPM = srows[:, 0].astype(int); SEX = srows[:, 1:3].astype(np.float32); SY = srows[:, 3:5].astype(np.float32)
print(f"{len(SY)} synthetic labelled points ({time.time()-T0:.0f}s)")

synthetic pretraining pool: 361 AIOMFAC-decomposable molecules of 361 candidates (BIMOG molecules excluded)


7220 synthetic labelled points (71s)


## 3) GNN encoder, training loop and models B–G

In [6]:
ATOMS = ["C", "O", "N", "Cl", "Br", "F", "I", "S"]
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
def graph(smi):
    mol = Chem.MolFromSmiles(smi)
    x = [[float(a.GetSymbol() == s) for s in ATOMS] + [a.GetDegree() / 4, a.GetTotalNumHs() / 4, float(a.GetIsAromatic()),
         float(a.IsInRing()), a.GetFormalCharge()] for a in mol.GetAtoms()]
    ei, ea = [], []
    for b in mol.GetBonds():
        f = [float(b.GetBondType() == t) for t in BONDS]; i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); ei += [(i, j), (j, i)]; ea += [f, f]
    if not ei: ei, ea = [(0, 0)], [[0.0] * 4]
    return np.array(x, np.float32), np.array(ei, np.int64).T, np.array(ea, np.float32)
class Bank:
    def __init__(self, graphs):
        xs, eis, eas, own, off = [], [], [], [], 0
        for k, (x, ei, ea) in enumerate(graphs):
            xs.append(x); eis.append(ei + off); eas.append(ea); own.append(np.full(len(x), k)); off += len(x)
        self.x = torch.tensor(np.concatenate(xs)); self.ei = torch.tensor(np.concatenate(eis, 1)); self.ea = torch.tensor(np.concatenate(eas))
        self.own = torch.tensor(np.concatenate(own)); self.n = len(graphs)
class GINE(nn.Module):
    def __init__(self, h):
        super().__init__(); self.mlp = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
    def forward(self, h, ei, e): return self.mlp(h + torch.zeros_like(h).index_add_(0, ei[1], torch.relu(h[ei[0]] + e)))
class Encoder(nn.Module):
    def __init__(self, h=64, emb=64, L=3):
        super().__init__()
        self.ap = nn.Linear(len(ATOMS) + 5, h); self.ep = nn.ModuleList([nn.Linear(4, h) for _ in range(L)])
        self.cv = nn.ModuleList([GINE(h) for _ in range(L)]); self.bn = nn.ModuleList([nn.BatchNorm1d(h) for _ in range(L)]); self.out = nn.Linear(2 * h, emb)
    def forward(self, b):
        h = self.ap(b.x)
        for c, n, e in zip(self.cv, self.bn, self.ep): h = torch.relu(n(c(h, b.ei, e(b.ea))))
        s = torch.zeros(b.n, h.shape[1]).index_add_(0, b.own, h); cnt = torch.zeros(b.n).index_add_(0, b.own, torch.ones(len(h))).clamp(min=1)
        return self.out(torch.cat([s / cnt[:, None], s], 1))

GB = {i: graph(POOL[i]["smiles"]) for i in bids}; GS = [graph(o["smiles"]) for o in SYN]
train_mols = np.unique(PM2[tr_all]); BANK_TR = Bank([GB[i] for i in train_mols]); BANK_ALL = Bank([GB[i] for i in bids])
LOC_TR = {m: k for k, m in enumerate(train_mols)}; LOC_ALL = {m: k for k, m in enumerate(bids)}
loc_tr = torch.tensor([LOC_TR.get(m, -1) for m in PM2]); loc_all = torch.tensor([LOC_ALL[m] for m in PM2])

def train_graph_model(enc, seed, use_desc=False, train_encoder=True, drop=0.0, max_epochs=200, patience=15, batch=128,
                      pts=None, Ymat=None, EX=None, PMv=None, bank_tr=None, loc_t=None, val=None, extra_dim=0):
    '''Generic trainer: head on [GNN embedding (trainable or frozen) | optional R_desc | (w, T)].'''
    pts = fit_pts if pts is None else pts; Ymat = Y2 if Ymat is None else Ymat; EX = EX2 if EX is None else EX
    PMv = PM2 if PMv is None else PMv; bank_tr = BANK_TR if bank_tr is None else bank_tr; loc_t = loc_tr if loc_t is None else loc_t
    val = val_pts if val is None else val
    torch.manual_seed(1000 + seed); rng = np.random.RandomState(seed)
    D = DESC[PMv] if use_desc else np.zeros((len(PMv), 0), np.float32)
    Z = np.concatenate([D, EX], 1); zm, zs = Z[pts].mean(0), Z[pts].std(0); zs = np.where(zs < 1e-3, 1.0, zs)
    Zt = torch.tensor(((Z - zm) / zs).astype(np.float32)); ym, ys = Ymat[pts].mean(0), Ymat[pts].std(0) + 1e-6; Yn = torch.tensor((Ymat - ym) / ys)
    head = Head(64 + Z.shape[1], L=2, drop=drop)
    params = list(head.parameters()) + (list(enc.parameters()) if train_encoder else [])
    opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    if not train_encoder:
        enc.eval()
        with torch.no_grad(): EMB_TR = enc(bank_tr)
    def fwd(bank, loc, idx, training):
        if train_encoder: enc.train(training); emb = enc(bank)
        else: emb = EMB_TR if bank is bank_tr else EMB_EVAL
        return head(torch.cat([emb[loc[idx]], Zt[idx]], 1))
    best, st, bad = 1e9, None, 0; val_t = torch.tensor(val)
    for ep in range(max_epochs):
        head.train(); perm = torch.from_numpy(pts[rng.permutation(len(pts))])
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(fwd(bank_tr, loc_t, i, True), Yn[i]).backward(); opt.step()
        head.eval()
        with torch.no_grad(): v = L(fwd(bank_tr, loc_t, val_t, False), Yn[val_t]).item()
        if v < best - 1e-5:
            best, bad = v, 0; st = ({k: t.clone() for k, t in head.state_dict().items()}, {k: t.clone() for k, t in enc.state_dict().items()})
        else:
            bad += 1
            if bad >= patience: break
    head.load_state_dict(st[0]); enc.load_state_dict(st[1]); head.eval(); enc.eval()
    return head, enc, (zm, zs, ym, ys, Zt), ep + 1

def evaluate(head, enc, norm):
    zm, zs, ym, ys, Zt = norm
    with torch.no_grad():
        P = head(torch.cat([enc(BANK_ALL)[loc_all], Zt], 1)).numpy() * ys + ym
    E = np.abs(P - Y2)
    return {"gen_w": E[gen_pts, 0].mean(), "gen_org": E[gen_pts, 1].mean(), "interp_org": E[interp_pts, 1].mean()}

def pretrain(seed):
    '''Encoder + head trained on the synthetic pool only (95/5 split); returns the encoder.'''
    p = np.random.RandomState(1).permutation(len(SY)); n5 = max(1, int(0.05 * len(SY)))
    bank_s = Bank(GS); loc_s = torch.tensor(SPM)
    enc = Encoder(); torch.manual_seed(2000 + seed)
    _, enc, _, ep = train_graph_model(enc, seed, max_epochs=100, patience=10, batch=256, pts=p[n5:], Ymat=SY, EX=SEX, PMv=SPM,
                                      bank_tr=bank_s, loc_t=loc_s, val=p[:n5])
    return enc, ep

In [7]:
import copy
for seed in SEEDS:
    t0 = time.time()
    torch.manual_seed(seed); h, e, nrm, ep = train_graph_model(Encoder(), seed)
    res.append({"model": "B", "seed": seed, **evaluate(h, e, nrm), "epochs": ep})
    enc_pre, ep_pre = pretrain(seed)
    h, e, nrm, ep = train_graph_model(copy.deepcopy(enc_pre), seed)
    res.append({"model": "C", "seed": seed, **evaluate(h, e, nrm), "epochs": ep})
    torch.manual_seed(seed); h, e, nrm, ep = train_graph_model(Encoder(), seed, use_desc=True)
    res.append({"model": "D", "seed": seed, **evaluate(h, e, nrm), "epochs": ep})
    h, e, nrm, ep = train_graph_model(copy.deepcopy(enc_pre), seed, use_desc=True)
    res.append({"model": "E", "seed": seed, **evaluate(h, e, nrm), "epochs": ep})
    for nm, enc0 in [("F", copy.deepcopy(enc_pre)), ("G", (torch.manual_seed(3000 + seed), Encoder())[1])]:
        enc0.eval()
        with torch.no_grad(): EMB_EVAL = enc0(BANK_ALL)
        h, e, nrm, ep = train_graph_model(enc0, seed, use_desc=True, train_encoder=False, drop=0.2)
        res.append({"model": nm, "seed": seed, **evaluate(h, e, nrm), "epochs": ep})
    print(f"  seed {seed}: pretraining {ep_pre} epochs; " + ", ".join(f"{r['model']} {r['gen_org']:.3f}" for r in res if r["seed"] == seed)
          + f" ({time.time()-t0:.0f}s; total {time.time()-T0:.0f}s)")
R3 = pd.DataFrame(res); R3.to_csv("table3_all_models_runs.csv", index=False)
order = ["A_emb", "A_desc", "B", "C", "D", "E", "F", "G"]
tab = R3.groupby("model")[["gen_w", "gen_org", "interp_org"]].agg(fmt).loc[order]
display(tab)
# per-seed comparison B vs C (pretraining effect)
bc = R3.pivot(index="seed", columns="model", values="gen_org")
print("C better than B in", int((bc["C"] < bc["B"]).sum()), "of", len(bc), "seeds; G vs F:", int((bc["G"] <= bc["F"]).sum()), "seeds with G ≤ F")

  seed 0: pretraining 51 epochs; A_emb 0.184, A_desc 0.316, B 0.363, C 0.471, D 0.293, E 0.315, F 0.593, G 0.380 (310s; total 381s)


  seed 1: pretraining 54 epochs; A_emb 0.179, A_desc 0.355, B 0.453, C 0.495, D 0.361, E 0.327, F 0.575, G 0.393 (239s; total 620s)


  seed 2: pretraining 50 epochs; A_emb 0.185, A_desc 0.283, B 0.344, C 0.475, D 0.266, E 0.315, F 0.645, G 0.325 (290s; total 910s)


  seed 3: pretraining 35 epochs; A_emb 0.166, A_desc 0.337, B 0.415, C 0.361, D 0.252, E 0.323, F 0.593, G 0.400 (334s; total 1244s)


  seed 4: pretraining 57 epochs; A_emb 0.190, A_desc 0.321, B 0.449, C 0.371, D 0.363, E 0.305, F 0.596, G 0.354 (267s; total 1511s)


,gen_w,gen_org,interp_org
model,,,
A_emb,0.021 ± 0.001,0.181 ± 0.009,0.085 ± 0.006
A_desc,0.032 ± 0.004,0.322 ± 0.027,0.088 ± 0.008
B,0.040 ± 0.004,0.405 ± 0.050,0.163 ± 0.026
C,0.039 ± 0.005,0.435 ± 0.063,0.165 ± 0.037
D,0.029 ± 0.003,0.307 ± 0.052,0.128 ± 0.028
E,0.032 ± 0.003,0.317 ± 0.008,0.128 ± 0.019
F,0.063 ± 0.012,0.600 ± 0.026,0.370 ± 0.049
G,0.038 ± 0.002,0.370 ± 0.031,0.242 ± 0.032


C better than B in 2 of 5 seeds; G vs F: 5 seeds with G ≤ F
